In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"

df = pd.read_csv(url)

df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [4]:
df = df[['label', 'tweet']]
df.columns = ['label', 'text']

df.head()

,label,text
0,0,@user when a father is dysfunctional and is s...
1,0,@user @user thanks for #lyft credit i can't us...
2,0,bihday your majesty
3,0,#model i love u take with u all the time in ...
4,0,factsguide: society now #motivation


In [8]:
df['label'] = df['label'].map({0: 'negative', 1: 'positive'})

In [ ]:

# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



# 2. LOAD DATASET (NO DOWNLOAD NEEDED)

url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"
df = pd.read_csv(url)

# Fix columns
df = df[['label', 'tweet']]
df.columns = ['label', 'text']

# Convert labels
df['label'] = df['label'].map({0: 'negative', 1: 'positive'})

print("Dataset Loaded Successfully!")
print(df.head())


 3. DATA UNDERSTANDING

print("\nShape:", df.shape)
print("\nClass Distribution:\n", df['label'].value_counts())
print("\nSample Text:\n", df['text'].iloc[0])



# 4. NLP PREPROCESSING

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text

def preprocess_text(text):
    text = clean_text(text)
    tokens = text.split()  # tokenization
    tokens = [word for word in tokens if word not in stop_words]  # stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # lemmatization
    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess_text)

print("\nPreprocessing Done!")
print(df[['text', 'clean_text']].head())



# 5. TRAIN-TEST SPLIT

X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# 6. FEATURE ENGINEERING


# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)



# 7. EVALUATION FUNCTION

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))



# 8. MODEL TRAINING


# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
evaluate_model("Logistic Regression (TF-IDF)", lr, X_test_tfidf, y_test)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
evaluate_model("Naive Bayes (BoW)", nb, X_test_bow, y_test)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
evaluate_model("Decision Tree (TF-IDF)", dt, X_test_tfidf, y_test)



# 9. FINAL PIPELINE

print("\nNLP PIPELINE:")
print("Raw Text → Cleaning → Preprocessing → Vectorization → Model → Evaluation")



# 10. SAMPLE PREDICTION

sample = ["I love this product, it's amazing!"]

sample_clean = [preprocess_text(text) for text in sample]
sample_vec = tfidf.transform(sample_clean)

prediction = lr.predict(sample_vec)

print("\nSample Prediction:", prediction[0])